<a href="https://colab.research.google.com/github/eliasrbarbour/AAI614_eliasbarbour/blob/main/Project1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Data Loading

In [35]:
!pip install -q ucimlrepo

import pandas as pd
import numpy as np
from ucimlrepo import fetch_ucirepo

adult = fetch_ucirepo(id=2)
X = adult.data.features
y = adult.data.targets

df = pd.concat([X, y], axis=1)
df.columns = df.columns.str.replace("-", "_")
df = df.rename(columns={"income": "gross_income_group"})

print(df.shape)
df.head()


(48842, 15)


,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,gross_income_group
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


Data Exploration

In [36]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   age                 48842 non-null  int64 
 1   workclass           47879 non-null  object
 2   fnlwgt              48842 non-null  int64 
 3   education           48842 non-null  object
 4   education_num       48842 non-null  int64 
 5   marital_status      48842 non-null  object
 6   occupation          47876 non-null  object
 7   relationship        48842 non-null  object
 8   race                48842 non-null  object
 9   sex                 48842 non-null  object
 10  capital_gain        48842 non-null  int64 
 11  capital_loss        48842 non-null  int64 
 12  hours_per_week      48842 non-null  int64 
 13  native_country      48568 non-null  object
 14  gross_income_group  48842 non-null  object
dtypes: int64(6), object(9)
memory usage: 5.6+ MB


**Q1**.Check the columns of your data. Are they the expected data types based on their descriptions in this text file description of the data?

In [37]:
# Check the values of the two binary columns
print(df["sex"].value_counts(), "\n")
print(df["gross_income_group"].value_counts())

sex
Male      32650
Female    16192
Name: count, dtype: int64 

gross_income_group
<=50K     24720
<=50K.    12435
>50K       7841
>50K.      3846
Name: count, dtype: int64


In [38]:
# Remove the trailing period from the test-file labels
df["gross_income_group"] = df["gross_income_group"].str.rstrip(".")
df["gross_income_group"].value_counts()

,count
gross_income_group,
<=50K,37155
>50K,11687


**Answer**: All six integer columns (age, fnlwgt, education_num, capital_gain, capital_loss, hours_per_week) were loaded as int64, and all categorical and binary columns were loaded as object (text), so the data types match the dataset description. However, checking the values of the two binary columns showed a problem: sex has the expected two values, but gross_income_group had four: <=50K, >50K, <=50K. and >50K.. The labels with a trailing period come from the test file (16,281 rows), which is combined with the training file (32,561 rows) to give the full 48,842 rows. I removed the trailing period with str.rstrip("."), which leaves two values: <=50K (37,155 rows) and >50K (11,687 rows). I also noticed that education and education_num carry the same information, since education_num is a numeric code for the education level.

**Q2**.Check the columns of your data. Are they the expected data types based on their descriptions in this text file description of the data?

In [46]:
# Show which columns the dataset description says have missing values
adult.variables[["name", "missing_values"]]

,name,missing_values
0,age,no
1,workclass,yes
2,fnlwgt,no
3,education,no
4,education-num,no
5,marital-status,no
6,occupation,yes
7,relationship,no
8,race,no
9,sex,no


In [47]:
# Count the NaN values in each column (True = missing, sum counts the Trues)
df.isna().sum()

,0
age,0
workclass,2799
fnlwgt,0
education,0
education_num,0
marital_status,0
occupation,2809
relationship,0
race,0
sex,0


In [48]:
# Keep only the text columns, since "?" can only appear in text
obj = df.select_dtypes("object")

# For each column: remove spaces, mark cells equal to "?", then count them
obj.apply(lambda s: s.str.strip().eq("?")).sum()

,0
workclass,0
education,0
marital_status,0
occupation,0
relationship,0
race,0
sex,0
native_country,0
gross_income_group,0


In [49]:
# Replace any cell that contains only "?" (with or without spaces) by NaN
df = df.replace(r"^\s*\?\s*$", np.nan, regex=True)

# Build a table with the number and percentage of missing values per column
missing = pd.DataFrame({
    "missing_count": df.isna().sum(),                  # number of missing values
    "missing_pct": (df.isna().mean() * 100).round(2)   # share of missing values, in %
})

# Show only the columns that have at least one missing value
missing[missing["missing_count"] > 0]

,missing_count,missing_pct
workclass,2799,5.73
occupation,2809,5.75
native_country,857,1.75


In [50]:
# Select people with a missing occupation but a known workclass,
# then count which workclass they belong to
df.loc[df["occupation"].isna() & df["workclass"].notna(), "workclass"].value_counts()

,count
workclass,
Never-worked,10
